# **R-learner: DML для функции эффекта**

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school), урок «Мета-обучатели: S, T и X», шаг «R-learner: DML для функции».

В уроке R-learner ([Nie и Wager, 2017](https://arxiv.org/abs/1712.04912)) был записан формулой: из исхода и воздействия вычитают их прогнозы по признакам (остатки Робинсона), а функцию эффекта $\tau(x)$ ищут взвешенной регрессией псевдоисхода $\tilde Y_i/\tilde T_i$ с весами $\tilde T_i^2$. Здесь мы соберем его руками и проверим три утверждения урока:

1. ортогонализация позволяет учить $\tau$ простой моделью с сильной регуляризацией;
2. схема ломается при сильном дисбалансе групп, потому что вес контрольных объектов $\tilde T_i^2$ близок к нулю;
3. R-learner дает еще и критерий выбора модели эффекта, который не требует знать истинный эффект.

План тетради:

1. данные IHDP с известным истинным эффектом;
2. остатки Робинсона с cross-fitting;
3. R-learner как взвешенная регрессия псевдоисхода;
4. сравнение с S-, T-, X- и DR-learner;
5. чувствительность к регуляризации модели эффекта;
6. где схема ломается: модель склонности и дисбаланс групп;
7. функция потерь R-learner как критерий выбора модели;
8. библиотечный вызов `econml.dml.NonParamDML`.

Тетрадь идет на процессоре; весь прогон занимает около минуты.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dr import DRLearner
from econml.dml import NonParamDML

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # ложные предупреждения numpy на некоторых процессорах
SEED = 0
pd.set_option('display.width', 140)

## 1. Данные IHDP

IHDP (Infant Health and Development Program) — рандомизированный эксперимент с недоношенными детьми с низким весом при рождении: воздействие — визиты специалисток на дом и развивающий детский сад. [Hill (2011)](https://doi.org/10.1198/jcgs.2010.08162) сделала из него полусинтетический набор, который стал стандартным для проверки оценок CATE:

- признаки настоящие: 25 характеристик ребенка и матери, 6 непрерывных (стандартизованы) и 19 бинарных;
- из группы воздействия удалена неслучайная часть детей, поэтому группы больше не сопоставимы: осталось 747 детей, из них 139 в группе воздействия;
- исходы сгенерированы по известной формуле (поверхность отклика B у Hill), поэтому у каждого ребенка известны оба ожидаемых потенциальных исхода $\mu_0(x)$ и $\mu_1(x)$ и истинный эффект $\tau(x)=\mu_1(x)-\mu_0(x)$.

Берем первую из десяти реализаций, которые выложили авторы CEVAE ([Louizos и др., 2017](https://github.com/AMLab-Amsterdam/CEVAE)). 70% детей идут на обучение, 30% — на отложенную проверку; разбиение стратифицировано по воздействию.

Ошибку оценки меряем так же, как Hill: $\mathrm{PEHE}=\sqrt{\tfrac1n\sum_i(\hat\tau(x_i)-\tau(x_i))^2}$ на отложенной части. На реальных данных такой метрики нет — в разделе 7 мы посмотрим, чем ее заменить.

In [ ]:
URL = 'https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/datasets/IHDP/csv/ihdp_npci_1.csv'
cols = ['treatment', 'y_factual', 'y_cfactual', 'mu0', 'mu1'] + [f'x{i}' for i in range(1, 26)]
df = pd.read_csv(URL, header=None, names=cols)

X = df[[f'x{i}' for i in range(1, 26)]].values
T = df.treatment.values.astype(int)
Y = df.y_factual.values
tau = (df.mu1 - df.mu0).values                 # истинный CATE: известен только потому, что данные полусинтетические

tr, te = train_test_split(np.arange(len(df)), test_size=0.3, random_state=SEED, stratify=T)
print('детей:', len(df), '| в группе воздействия:', T.sum(), f'({T.mean():.3f})')
print('обучение:', len(tr), '| проверка:', len(te))
print(f'истинный эффект: среднее {tau.mean():.2f}, стандартное отклонение {tau.std():.2f}')


def pehe(est):
    # ошибка оценки CATE на отложенной части
    return np.sqrt(np.mean((est - tau[te]) ** 2))


def gb(depth=3):
    # модель исхода, общая для всех схем тетради
    return GradientBoostingRegressor(n_estimators=200, max_depth=depth, learning_rate=0.05, random_state=SEED)

## 2. Остатки Робинсона с cross-fitting

Первый шаг R-learner — две вспомогательные модели: модель среднего исхода $\hat m(x)\approx\mathbb E[Y\mid X=x]$ по всей выборке, без разделения на группы, и модель склонности $\hat e(x)\approx\mathbb P(T=1\mid X=x)$. Остатки объекта считают моделями, которые этого объекта не видели (cross-fitting): иначе переобученная $\hat m$ «съест» часть эффекта вместе с шумом.

**Задание 1. На обучающей части посчитайте прогнозы `m_hat` и `e_hat` функцией `cross_val_predict` с разбиением `kf = KFold(5, shuffle=True, random_state=SEED)`: модель исхода — `gb()`, модель склонности — `LogisticRegression(max_iter=1000)` (вероятности — `method='predict_proba'`, столбец 1). Посчитайте остатки `y_res` $=\tilde Y$ и `t_res` $=\tilde T$ и выведите, какая доля суммарного веса $\sum_i\tilde T_i^2$ приходится на объекты группы воздействия обучающей части.**

In [ ]:
# Ваш код здесь

Распределение веса $\tilde T_i^2$ по двум группам. У контрольного объекта вес равен $\hat e(x)^2$, у объекта группы воздействия — $(1-\hat e(x))^2$: при склонности около $0.2$ это примерно $0.04$ против $0.64$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
ax1.hist(e_hat[T[tr] == 0], bins=30, alpha=0.6, color='#3b6ea5', label='control')
ax1.hist(e_hat[T[tr] == 1], bins=30, alpha=0.6, color='#e67e22', label='treated')
ax1.set_xlabel('Propensity ê(x), cross-fitted')
ax1.set_ylabel('Objects')
ax1.set_title('Propensity by group')
ax1.legend()
ax2.hist(w[T[tr] == 0], bins=30, alpha=0.6, color='#3b6ea5', label='control')
ax2.hist(w[T[tr] == 1], bins=30, alpha=0.6, color='#e67e22', label='treated')
ax2.set_xlabel('R-learner weight (T − ê(x))²')
ax2.set_title('Weight by group')
ax2.legend()
fig.tight_layout()
plt.show()

## 3. R-learner как взвешенная регрессия псевдоисхода

Функция потерь R-learner $\frac1n\sum_i(\tilde Y_i-\tau(X_i)\tilde T_i)^2$ переписывается как $\frac1n\sum_i\tilde T_i^2\bigl(\tilde Y_i/\tilde T_i-\tau(X_i)\bigr)^2$. Значит, подойдет любой обучатель, который принимает `sample_weight`: учим его на признаках $X$ предсказывать псевдоисход $\tilde Y_i/\tilde T_i$ с весами $\tilde T_i^2$.

Модель эффекта берем нарочно простую — линейную модель со штрафом лассо: урок говорит, что сложность исхода забирает $\hat m$, а для $\tau$ хватает простой модели с сильной регуляризацией.

**Задание 2. Напишите функцию `r_learner(final)`, которая обучает модель `final` на `X[tr]` с псевдоисходом `y_res / t_res` и весами `t_res ** 2` и возвращает обученную модель. Обучите `Lasso(alpha=0.05)`, сохраните оценку на отложенной части в `tau_r` и выведите ее PEHE и число ненулевых коэффициентов.**

In [ ]:
# Ваш код здесь

## 4. Сравнение со схемами урока

Те же данные, та же модель исхода `gb()` внутри всех схем: S-, T- и X-learner из урока и DR-learner — схема, в которой псевдоисход строится дважды робастной оценкой (тот же псевдоисход $\varphi_i$, что в шаге о policy learning урока «Как читать оценку и где ошибиться»). Для каждой схемы печатаем PEHE и ошибку среднего эффекта на отложенной части.

In [ ]:
logit = lambda: LogisticRegression(max_iter=1000)
est = {
    'S-learner': SLearner(overall_model=gb()).fit(Y[tr], T[tr], X=X[tr]).effect(X[te]),
    'T-learner': TLearner(models=gb()).fit(Y[tr], T[tr], X=X[tr]).effect(X[te]),
    'X-learner': XLearner(models=gb(), propensity_model=logit()).fit(Y[tr], T[tr], X=X[tr]).effect(X[te]),
    'DR-learner': DRLearner(model_regression=gb(), model_propensity=logit(), model_final=gb(),
                            cv=5, random_state=SEED).fit(Y[tr], T[tr], X=X[tr]).effect(X[te]),
    'R-learner (лассо)': tau_r,
}
table = pd.DataFrame({'PEHE ↓': {k: pehe(v) for k, v in est.items()},
                      'ошибка среднего эффекта': {k: v.mean() - tau[te].mean() for k, v in est.items()}})
print(table.round(3))

Рассеяние «истинный эффект против оцененного» на отложенной части: красная линия — $y=x$. Облако, вытянутое вдоль горизонтали, значит, что схема почти не различает детей с большим и малым эффектом; облако, разбросанное по вертикали, — что она шумит.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8), sharex=True, sharey=True)
lim = [tau[te].min() - 0.5, tau[te].max() + 0.5]
for ax, (name, e) in zip(axes, est.items()):
    ax.scatter(tau[te], e, s=8, alpha=0.5, color='#3b6ea5')
    ax.plot(lim, lim, color='#c0392b', lw=1.2)
    ax.set_title(f'{name}\nPEHE = {pehe(e):.2f}')
    ax.set_xlabel('True effect τ(x)')
axes[0].set_ylabel('Estimated effect τ̂(x)')
axes[0].set_ylim(-2, 10)
fig.tight_layout()
plt.show()

## 5. Чувствительность к регуляризации модели эффекта

Урок называет главным гиперпараметром R-learner регуляризацию $\Lambda(\tau)$. Проверим это на двух классах модели эффекта: лассо с весом штрафа `alpha` от $10^{-3}$ до $1$ и бустинг глубины 1, 2 и 3 (200 деревьев). Остатки `y_res`, `t_res` при этом не меняются: меняется только последний шаг.

In [ ]:
alphas = np.logspace(-3, 0, 13)
lasso_pehe, lasso_nnz = [], []
for a in alphas:
    m = r_learner(Lasso(alpha=a))
    lasso_pehe.append(pehe(m.predict(X[te])))
    lasso_nnz.append((m.coef_ != 0).sum())
gb_pehe = {d: pehe(r_learner(gb(d)).predict(X[te])) for d in (1, 2, 3)}

print(pd.DataFrame({'alpha': alphas, 'PEHE': lasso_pehe, 'ненулевых коэффициентов': lasso_nnz}).round(3).to_string(index=False))
print('бустинг как модель эффекта:', {f'глубина {d}': round(float(v), 3) for d, v in gb_pehe.items()})

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.semilogx(alphas, lasso_pehe, 'o-', color='#3b6ea5', label='R-learner, lasso final model')
for d, v in gb_pehe.items():
    ax.axhline(v, ls='--', lw=1, color=['#e67e22', '#c0392b', '#7f8c8d'][d - 1], label=f'R-learner, boosting depth {d}')
ax.axhline(pehe(est['S-learner']), ls=':', color='black', label='S-learner')
ax.set_xlabel('Lasso penalty alpha')
ax.set_ylabel('PEHE on held-out part ↓')
ax.set_title('R-learner: error versus regularization of τ')
ax.legend(fontsize=8)
plt.show()

**Задание 3. По выводу ячейки выше: при каком значении `alpha` из перебора PEHE лассо минимальна и сколько ненулевых коэффициентов остается у модели эффекта при этом значении? Почему бустинг глубины 3 как модель эффекта ошибается сильнее, чем та же модель в роли $\hat m$?**

**Ваш ответ:**

## 6. Где схема ломается

### 6.1. Модель склонности

Псевдоисход делится на $\tilde T_i=T_i-\hat e(X_i)$. Если модель склонности уверенно предсказывает $\hat e\approx1$ для объекта группы воздействия или $\hat e\approx0$ для контрольного, остаток близок к нулю, а псевдоисход — огромен. Логистическая регрессия дает гладкие вероятности; бустинг-классификатор по умолчанию подгоняется сильнее. Сравним R-learner с двумя моделями склонности при той же модели исхода.

In [ ]:
e_gb = cross_val_predict(GradientBoostingClassifier(random_state=SEED), X[tr], T[tr], cv=kf, method='predict_proba')[:, 1]
t_res_gb = T[tr] - e_gb
pseudo = {'логистическая регрессия': y_res / t_res, 'бустинг-классификатор': y_res / t_res_gb}
for name, ps in pseudo.items():
    print(f'{name:>24}: max |псевдоисход| = {np.abs(ps).max():8.1f}')

rows = {}
for name, tr_res in (('логистическая регрессия', t_res), ('бустинг-классификатор', t_res_gb)):
    for fname, final in (('лассо, alpha=0.05', Lasso(alpha=0.05)), ('бустинг глубины 1', gb(1))):
        m = final.fit(X[tr], y_res / tr_res, sample_weight=tr_res ** 2)
        rows[(name, fname)] = pehe(m.predict(X[te]))
print(pd.Series(rows).unstack().round(3))

### 6.2. Дисбаланс групп

Урок предсказывает: при сопоставимых группах выигрывает ортогонализация R-learner, при сильном дисбалансе — смешивание X-learner. На IHDP доля группы воздействия одна, поэтому возьмем синтетические данные в духе Nie и Wager: сложный исход, простой эффект и доля группы воздействия, которую мы задаем сами.

- Шесть признаков, $X\sim U[0,1]^6$.
- Исход без воздействия: $\mu_0(x)=4\sin(\pi x_0x_1)+16(x_2-0.5)^2+2x_3+x_4$.
- Эффект простой и линейный: $\tau(x)=(x_0+x_1)/2$.
- Воздействие получает случайная доля объектов $p$, $Y=\mu_0(X)+T\tau(X)+\varepsilon$, $\varepsilon\sim\mathcal N(0,0.5^2)$.

Для каждого $p\in\{0.5;\ 0.2;\ 0.1;\ 0.05\}$ и трех сидов (0, 1, 2) генерируем обучающую выборку из 2000 объектов и проверочную из 2000 и сравниваем PEHE трех оценок: R-learner с лассо, R-learner с бустингом глубины 2 и X-learner.

**Задание 4. Допишите функцию `imbalance_run(p, seed)`: она генерирует обе выборки функцией `simulate`, считает остатки Робинсона на обучающей выборке (cross-fitting как в задании 1), обучает R-learner с `Lasso(alpha=0.01)` и с `gb(2)`, X-learner с `gb()` и `LogisticRegression(max_iter=1000)` и возвращает словарь PEHE на проверочной выборке. Соберите таблицу средних по трем сидам.**

In [ ]:
def simulate(n, p, rng):
    Xs = rng.uniform(0, 1, (n, 6))
    mu0 = 4 * np.sin(np.pi * Xs[:, 0] * Xs[:, 1]) + 16 * (Xs[:, 2] - 0.5) ** 2 + 2 * Xs[:, 3] + Xs[:, 4]
    ts = (Xs[:, 0] + Xs[:, 1]) / 2
    Ts = rng.binomial(1, p, n)
    return Xs, Ts, mu0 + Ts * ts + rng.normal(0, 0.5, n), ts

In [ ]:
# Ваш код здесь

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
for (name, col), color in zip(imb.items(), ['#3b6ea5', '#c0392b', '#e67e22']):
    ax.plot(imb.index, col, 'o-', color=color, label=name)
ax.set_xscale('log')
ax.set_xticks(shares)
ax.set_xticklabels([str(s) for s in shares])
ax.invert_xaxis()
ax.set_xlabel('Treated share p (right = stronger imbalance)')
ax.set_ylabel('PEHE, mean of 3 seeds ↓')
ax.set_title('Synthetic data: complex outcome, linear effect')
ax.legend()
plt.show()

**Задание 5. Сравните строки таблицы задания 4. У какой из трех оценок PEHE растет быстрее всего при переходе от $p=0.5$ к $p=0.05$, и подтверждают ли числа прогноз урока «при сильном дисбалансе выигрывает X-learner» для обеих моделей эффекта?**

**Ваш ответ:**

## 7. Функция потерь R-learner как критерий выбора модели

На реальных данных PEHE не посчитать: истинного $\tau(x)$ нет. Но функция потерь R-learner $\frac1n\sum_i(\tilde Y_i-\hat\tau(X_i)\tilde T_i)^2$ требует только $Y$, $T$ и остатков, и ее можно посчитать для **любой** оценки $\hat\tau$ — в том числе для S-, T- и X-learner. [Nie и Wager](https://arxiv.org/abs/1712.04912) предлагают выбирать модель эффекта по этой потере на отложенных данных; в EconML так устроен `econml.score.RScorer`.

Считаем остатки на отложенной части с отдельным cross-fitting (модели, обученные только на ней), потом — потерю для каждой оценки из раздела 4 и для R-learner с бустингом глубины 3. Правильный критерий должен упорядочить оценки так же, как PEHE.

In [ ]:
kf_te = KFold(5, shuffle=True, random_state=SEED + 1)
y_res_te = Y[te] - cross_val_predict(gb(), X[te], Y[te], cv=kf_te)
t_res_te = T[te] - cross_val_predict(LogisticRegression(max_iter=1000), X[te], T[te], cv=kf_te, method='predict_proba')[:, 1]
r_loss = lambda e: np.mean((y_res_te - e * t_res_te) ** 2)

cand = dict(est)
cand['R-learner (gb depth 3)'] = r_learner(gb(3)).predict(X[te])
sel = pd.DataFrame({'PEHE ↓': {k: pehe(v) for k, v in cand.items()},
                    'R-loss ↓': {k: r_loss(v) for k, v in cand.items()}}).sort_values('PEHE ↓')
print(sel.round(4))
rho = spearmanr(sel['PEHE ↓'], sel['R-loss ↓']).statistic
print(f'ранговая корреляция Спирмена PEHE и R-loss по {len(sel)} оценкам: {rho:.3f}')

Потеря R-learner не равна PEHE даже с точностью до константы: в нее входит шум исхода и ошибки остатков, поэтому две оценки CATE с близкой PEHE она может переставить местами: здесь по функции потерь R-learner первой стоит оценка X-learner, третья по PEHE. Самую грубую ошибку — R-learner с бустингом глубины 3 — она ставит последней.

## 8. Библиотечный вызов

В EconML R-learner с произвольной моделью эффекта — `econml.dml.NonParamDML`: `model_y` — модель $\hat m$, `model_t` — модель склонности, `model_final` — модель эффекта (она обязана принимать `sample_weight`), `cv` — число фолдов cross-fitting. В CausalML та же схема — `causalml.inference.meta.BaseRRegressor`. Фолды внутри библиотеки другие, поэтому число совпадет с нашим не до третьего знака.

In [ ]:
npd = NonParamDML(model_y=gb(), model_t=LogisticRegression(max_iter=1000), model_final=Lasso(alpha=0.05),
                  discrete_treatment=True, cv=5, random_state=SEED).fit(Y[tr], T[tr], X=X[tr])
print(f'NonParamDML (лассо, alpha=0.05): PEHE {pehe(npd.effect(X[te])):.3f}  | наш R-learner: {pehe(tau_r):.3f}')

## Что унести из тетради

- **R-learner — две вспомогательные модели и одна взвешенная регрессия.** Вся ортогонализация — в остатках Робинсона с cross-fitting; модель эффекта видит только псевдоисход и веса $\tilde T_i^2$.
- **Регуляризация модели эффекта решает больше, чем выбор схемы.** На IHDP лассо с подобранным штрафом и бустинг глубины 3 дают ошибки оценки CATE, различающиеся в разы, при одних и тех же остатках.
- **Модель склонности — слабое место.** Уверенная модель склонности делает остатки $\tilde T_i$ почти нулевыми, псевдоисход взрывается, и гибкая модель эффекта подгоняется под выбросы.
- **Дисбаланс групп бьет по гибкой модели эффекта.** При малой доле группы воздействия вес держится на немногих объектах; простая модель эффекта это переносит, бустинг — нет, и тогда X-learner надежнее.
- **Потеря R-learner — критерий выбора без истинного эффекта.** Она не равна PEHE, но отсекает грубые ошибки и годится для подбора гиперпараметров на реальных данных.

Источники:
1. [Nie & Wager, 2017. Quasi-oracle estimation of heterogeneous treatment effects](https://arxiv.org/abs/1712.04912)
2. [Hill, 2011. Bayesian nonparametric modeling for causal inference](https://doi.org/10.1198/jcgs.2010.08162)
3. [Künzel et al., 2019. Metalearners for estimating heterogeneous treatment effects using machine learning](https://arxiv.org/abs/1706.03461)
4. [EconML](https://github.com/py-why/EconML), [CausalML](https://github.com/uber/causalml)
5. [Louizos, Shalit, Mooij & Sontag, 2017. Causal effect inference with deep latent-variable models](https://arxiv.org/abs/1705.08821); данные IHDP — первая из десяти реализаций в [репозитории CEVAE](https://github.com/AMLab-Amsterdam/CEVAE)